# Synergy Bootstrap Figures With Percentile Bands

This notebook creates non-interactive, publication-style figure components for the HT29 drug synergy bootstrap analysis. Curve bandwidths are defined by the pointwise 2.5% and 97.5% bootstrap quantiles, not by standard deviation.

The expected inputs are the CSV outputs from the DEPICT and observed-LINCS bootstrap runners.

## 1. Imports, Paths, and Style

The style and panel sizes are chosen to match the compact component-export logic used in the DEPICT grand-figure plotting notebook.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)


In [ ]:
def find_project_dir(start: Path | None = None) -> Path:
    """Find the DrugSynergyBootstrap project root by searching upward for data/."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError(
        'Could not find a project-level data/ folder. Run from the project root, '
        'from code/, or edit PROJECT_DIR manually.'
    )


PROJECT_DIR = Path("~/DEPICT/Code/downstream_analysis_code/DrugSynergy")
RESULTS_ROOT = PROJECT_DIR / 'results' / 'synergy_bootstrap'
FIGURE_DIR = RESULTS_ROOT / 'figures' / 'quantile_components_compact_metrics'

SOURCE_ORDER = ['depict', 'observed_lincs']
MODEL_ORDER = ['logistic_regression', 'random_forest']
EXPORT_FORMATS = ('pdf',)
EXPORT_DPI = 600

print(f'Project directory: {PROJECT_DIR}')
print(f'Results root:      {RESULTS_ROOT}')
print(f'Figure directory:  {FIGURE_DIR}')


In [ ]:
mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9.0,
    'axes.linewidth': 0.85,
    'axes.titlesize': 9.5,
    'axes.labelsize': 8.8,
    'xtick.labelsize': 8.0,
    'ytick.labelsize': 8.0,
    'legend.fontsize': 7.3,
    'legend.title_fontsize': 7.5,
    'legend.frameon': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.04,
})

SIZE_CURVE = (3.35, 3.0)
SIZE_METRIC = (3.35, 2.35)
SIZE_LEGEND = (3.8, 0.58)

SOURCE_LABELS = {
    'depict': 'DEPICT prediction',
    'observed_lincs': 'Observed LINCS proxy',
}

SOURCE_SHORT_LABELS = {
    'depict': 'DEPICT',
    'observed_lincs': 'Observed LINCS',
}

SOURCE_COLORS = {
    'depict': '#0072B2',
    'observed_lincs': '#D55E00',
}

MODEL_SHORT = {
    'logistic_regression': 'RLR',
    'random_forest': 'RF',
}

MODEL_LONG = {
    'logistic_regression': 'Regularized logistic regression',
    'random_forest': 'Random forest',
}

METRIC_LABELS = {
    'roc_auc': 'ROC-AUC',
    'pr_auc': 'PR-AUC',
    'accuracy': 'Accuracy',
    'macro_f1': 'Macro-F1',
}


## 2. Load Results

The notebook reads the bootstrap summary tables, original LOO metrics, and interpolated ROC/PR curve samples.

In [ ]:
def read_source_results(results_root: Path, source: str) -> dict[str, pd.DataFrame]:
    """Read all plotting inputs for one source."""
    source_dir = results_root / source
    paths = {
        'summary': source_dir / 'performance_summary.csv',
        'original': source_dir / 'original_loo_metrics.csv',
        'roc': source_dir / 'roc_curves.csv',
        'pr': source_dir / 'pr_curves.csv',
    }
    missing = [path for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(
            'Missing result files. Run the bootstrap analysis first. Missing:\n'
            + '\n'.join(str(path) for path in missing)
        )
    return {key: pd.read_csv(path) for key, path in paths.items()}


def load_all_results(results_root: Path, sources: Iterable[str]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    summary_frames = []
    original_frames = []
    roc_frames = []
    pr_frames = []

    for source in sources:
        tables = read_source_results(results_root, source)
        summary_frames.append(tables['summary'])
        original_frames.append(tables['original'])
        roc_frames.append(tables['roc'])
        pr_frames.append(tables['pr'])

    summary = pd.concat(summary_frames, ignore_index=True)
    original = pd.concat(original_frames, ignore_index=True)
    roc = pd.concat(roc_frames, ignore_index=True)
    pr = pd.concat(pr_frames, ignore_index=True)

    for table in (summary, original, roc, pr):
        table['source_label'] = table['source'].map(SOURCE_LABELS).fillna(table['source'])
        table['source_short'] = table['source'].map(SOURCE_SHORT_LABELS).fillna(table['source'])
        table['model_short'] = table['model'].map(MODEL_SHORT).fillna(table['model'])

    summary['metric_label'] = summary['metric'].map(METRIC_LABELS).fillna(summary['metric'])
    return summary, original, roc, pr


summary_df, original_df, roc_df, pr_df = load_all_results(RESULTS_ROOT, SOURCE_ORDER)
available_models = [model for model in MODEL_ORDER if model in set(summary_df['model'])]

print(f'Loaded sources: {sorted(summary_df["source"].unique())}')
print(f'Loaded models:  {available_models}')
summary_df.head()


In [ ]:
metric_table = (
    summary_df
    .loc[:, [
        'source_label', 'model_short', 'metric_label', 'original_loo',
        'bootstrap_mean', 'bootstrap_q025', 'bootstrap_q975', 'n_bootstrap_valid'
    ]]
    .rename(columns={
        'source_label': 'Input source',
        'model_short': 'Model',
        'metric_label': 'Metric',
        'original_loo': 'Original LOO',
        'bootstrap_mean': 'Bootstrap mean',
        'bootstrap_q025': 'Bootstrap 2.5%',
        'bootstrap_q975': 'Bootstrap 97.5%',
        'n_bootstrap_valid': 'Valid bootstrap samples',
    })
)

(RESULTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
metric_table.to_csv(RESULTS_ROOT / 'figures' / 'quantile_metric_table.csv', index=False)
metric_table


## 3. Plotting Helpers

Curve bands are computed pointwise as the 2.5% and 97.5% quantiles across bootstrap curves at each interpolation grid point. Scalar metric intervals use the same bootstrap quantiles from `performance_summary.csv`.

In [ ]:
def lighten_axes_grid(ax: plt.Axes, axis: str = 'both') -> None:
    """Apply a compact, light grid style consistent with the main DEPICT figures."""
    ax.set_axisbelow(True)
    ax.grid(True, axis=axis, color='#E8EAED', linewidth=0.65)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#4B5563')
    ax.spines['bottom'].set_color('#4B5563')
    ax.tick_params(width=0.75, length=3)


def save_component(fig: plt.Figure, stem: str) -> list[Path]:
    """Save a figure component in all requested manuscript formats."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    paths = []
    for fmt in EXPORT_FORMATS:
        path = FIGURE_DIR / f'{stem}.{fmt}'
        fig.savefig(path, dpi=EXPORT_DPI)
        paths.append(path)
    plt.close(fig)
    return paths


def summarize_curve(curve_df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    """Compute mean and pointwise percentile bands for one curve table."""
    return (
        curve_df
        .groupby(['source', 'source_short', 'source_label', 'model', 'model_short', x_col], sort=False)[y_col]
        .agg(
            mean='mean',
            q025=lambda values: values.quantile(0.025),
            q975=lambda values: values.quantile(0.975),
        )
        .reset_index()
    )


def get_metric_row(source: str, model: str, metric: str) -> pd.Series:
    rows = summary_df[(summary_df['source'] == source) & (summary_df['model'] == model) & (summary_df['metric'] == metric)]
    if rows.empty:
        raise KeyError(f'No metric row for source={source}, model={model}, metric={metric}')
    return rows.iloc[0]


def positive_prevalence(model: str) -> float | None:
    rows = original_df[original_df['model'] == model]
    if rows.empty or 'n_positive' not in rows.columns or 'n_pairs' not in rows.columns:
        return None
    n_positive = rows['n_positive'].iloc[0]
    n_pairs = rows['n_pairs'].iloc[0]
    return float(n_positive / n_pairs) if n_pairs else None


def interval_legend_label(row: pd.Series, metric: str) -> str:
    """Compact legend text: source plus metric mean and percentile interval."""
    return (
        f"{row['source_short']}: {row['bootstrap_mean']:.3f} "
        f"[{row['bootstrap_q025']:.3f}, {row['bootstrap_q975']:.3f}]"
    )


## 4. ROC and Precision-Recall Curve Components

Each model receives its own ROC and precision-recall component. This avoids overplotting four shaded bands in one axis.

In [ ]:
def plot_curve_component(model: str, curve_kind: str) -> plt.Figure:
    """Plot one ROC or PR component for one model with percentile bootstrap bands."""
    if curve_kind == 'roc':
        curve_summary = summarize_curve(roc_df[roc_df['model'] == model], 'fpr_grid', 'tpr')
        x_col, y_col = 'fpr_grid', 'tpr'
        metric = 'roc_auc'
        title = f"ROC curve: {MODEL_SHORT.get(model, model)}"
        x_label = 'False positive rate'
        y_label = 'True positive rate'
    elif curve_kind == 'pr':
        curve_summary = summarize_curve(pr_df[pr_df['model'] == model], 'recall_grid', 'precision')
        x_col, y_col = 'recall_grid', 'precision'
        metric = 'pr_auc'
        title = f"Precision-recall: {MODEL_SHORT.get(model, model)}"
        x_label = 'Recall'
        y_label = 'Precision'
    else:
        raise ValueError("curve_kind must be 'roc' or 'pr'.")

    fig, ax = plt.subplots(figsize=SIZE_CURVE)

    for source in SOURCE_ORDER:
        source_curve = curve_summary[curve_summary['source'] == source].sort_values(x_col)
        if source_curve.empty:
            continue
        row = get_metric_row(source, model, metric)
        color = SOURCE_COLORS[source]
        ax.plot(
            source_curve[x_col], source_curve['mean'],
            color=color, linewidth=2.0,
            label=interval_legend_label(row, metric),
        )
        ax.fill_between(
            source_curve[x_col], source_curve['q025'], source_curve['q975'],
            color=color, alpha=0.18, linewidth=0,
        )

    if curve_kind == 'roc':
        ax.plot([0, 1], [0, 1], color='#9CA3AF', linestyle='--', linewidth=0.85, zorder=0)
    else:
        prevalence = positive_prevalence(model)
        if prevalence is not None:
            ax.axhline(prevalence, color='#9CA3AF', linestyle='--', linewidth=0.85, zorder=0)
            ax.text(0.98, prevalence + 0.015, f'Prevalence = {prevalence:.2f}', ha='right', va='bottom', fontsize=7.2, color='#6B7280')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title, loc='left', fontweight='bold', pad=5)
    ax.legend(title=f"{METRIC_LABELS[metric]} mean [2.5%, 97.5%]", loc='lower right' if curve_kind == 'roc' else 'lower left')
    lighten_axes_grid(ax)
    fig.tight_layout(pad=0.45)
    return fig


curve_figures = {}
for model in available_models:
    for curve_kind in ('roc', 'pr'):
        fig = plot_curve_component(model, curve_kind)
        curve_figures[(model, curve_kind)] = fig
        display(fig)


## 5. Scalar Metric Components

Accuracy and Macro-F1 are shown as compact horizontal point-range plots with the same width as the curve panels. Points are bootstrap means, horizontal intervals are 2.5-97.5% bootstrap quantiles, and open diamonds show the original leave-one-out metric. Each point is labeled with `mean [2.5%, 97.5%]`.

In [ ]:
def metric_axis_limits(metric: str, lower_bound: float = 0.50, upper_bound: float = 1.00, pad: float = 0.025) -> tuple[float, float]:
    """Choose a focused axis range that preserves context while reducing blank space."""
    rows = summary_df[summary_df['metric'] == metric]
    low = min(rows['bootstrap_q025'].min(), rows['original_loo'].min()) - pad
    high = max(rows['bootstrap_q975'].max(), rows['original_loo'].max()) + pad
    low = max(lower_bound, np.floor(low / 0.05) * 0.05)
    high = min(upper_bound, np.ceil(high / 0.05) * 0.05)
    if high - low < 0.20:
        center = (high + low) / 2
        low = max(lower_bound, center - 0.10)
        high = min(upper_bound, center + 0.10)
    return float(low), float(high)


def annotate_metric_interval(
    ax: plt.Axes,
    mean: float,
    lower: float,
    upper: float,
    y: float,
    color: str,
    x_left: float,
    x_right: float,
) -> None:
    """Label one point-range estimate with mean and percentile interval."""
    x_range = x_right - x_left
    label = f'{mean:.3f}\n[{lower:.3f}, {upper:.3f}]'
    if (x_right - upper) >= 0.20 * x_range:
        x_text = upper + 0.018 * x_range
        ha = 'left'
    else:
        x_text = lower - 0.018 * x_range
        ha = 'right'
    ax.text(
        x_text, y, label,
        ha=ha, va='center', fontsize=6.65, color=color, linespacing=0.92,
        bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 0.6, 'alpha': 0.88},
        zorder=5,
    )


def plot_metric_component(metric: str) -> plt.Figure:
    """Plot one compact scalar metric component for RLR and RF using percentile intervals."""
    fig, ax = plt.subplots(figsize=SIZE_METRIC)
    y_base = np.arange(len(available_models))[::-1]
    source_offsets = {
        SOURCE_ORDER[0]: 0.105,
        SOURCE_ORDER[1]: -0.105,
    }
    x_left, x_right = metric_axis_limits(metric)

    for source in SOURCE_ORDER:
        for model_index, model in enumerate(available_models):
            row = get_metric_row(source, model, metric)
            y = y_base[model_index] + source_offsets[source]
            mean = row['bootstrap_mean']
            lower = row['bootstrap_q025']
            upper = row['bootstrap_q975']
            original = row['original_loo']
            color = SOURCE_COLORS[source]

            ax.errorbar(
                mean, y,
                xerr=[[mean - lower], [upper - mean]],
                fmt='o', markersize=5.0,
                color=color, ecolor=color,
                elinewidth=1.25, capsize=3.0, capthick=1.0,
                zorder=3,
            )
            ax.scatter(
                original, y,
                marker='D', s=22,
                facecolor='white', edgecolor=color,
                linewidth=0.85, zorder=4,
            )
            annotate_metric_interval(ax, mean, lower, upper, y, color, x_left, x_right)

    ax.set_yticks(y_base)
    ax.set_yticklabels([MODEL_SHORT.get(model, model) for model in available_models])
    ax.set_ylim(y_base.min() - 0.42, y_base.max() + 0.42)
    ax.set_xlim(x_left, x_right)
    ax.set_xlabel(METRIC_LABELS[metric])
    ax.set_title(METRIC_LABELS[metric], loc='left', fontweight='bold', pad=5)
    lighten_axes_grid(ax, axis='x')
    fig.tight_layout(pad=0.45)
    return fig


metric_figures = {}
for metric in ('accuracy', 'macro_f1'):
    fig = plot_metric_component(metric)
    metric_figures[metric] = fig
    display(fig)


## 6. Standalone Legend and Export

The following cell saves all displayed figure components as PDF files only. These files can be assembled into the final manuscript layout.

In [ ]:
def make_standalone_legend() -> plt.Figure:
    """Create a compact standalone legend for source colors and original LOO marker."""
    fig, ax = plt.subplots(figsize=SIZE_LEGEND)
    ax.axis('off')
    handles = [
        Line2D([0], [0], color=SOURCE_COLORS[source], marker='o', linewidth=1.7, markersize=5, label=SOURCE_LABELS[source])
        for source in SOURCE_ORDER
    ]
    handles.append(Line2D([0], [0], color='#4B5563', marker='D', markerfacecolor='white', linewidth=0, markersize=5, label='Original LOO'))
    ax.legend(handles=handles, loc='center', ncol=3, frameon=False, handletextpad=0.55, columnspacing=1.0)
    fig.tight_layout(pad=0.05)
    return fig


legend_fig = make_standalone_legend()
display(legend_fig)


In [ ]:
saved_paths = []

for (model, curve_kind), fig in curve_figures.items():
    stem = f"Synergy_{curve_kind.upper()}_{MODEL_SHORT.get(model, model)}_BootstrapQuantileBand"
    saved_paths.extend(save_component(fig, stem))

for metric, fig in metric_figures.items():
    stem = f"Synergy_{metric}_{'_'.join(MODEL_SHORT.get(model, model) for model in available_models)}_BootstrapQuantileInterval"
    saved_paths.extend(save_component(fig, stem))

saved_paths.extend(save_component(legend_fig, 'Synergy_SourceLegend_BootstrapQuantile'))

saved_index = pd.DataFrame({'path': [str(path) for path in saved_paths]})
saved_index.to_csv(FIGURE_DIR / 'saved_quantile_figure_components.csv', index=False)

print(f'Saved {len(saved_paths)} files to {FIGURE_DIR}')
saved_index


## Suggested Caption Text

Curves show the mean ROC or precision-recall curve across 1000 bootstrap resamples of leave-one-out prediction rows. Shaded bands indicate pointwise 2.5% to 97.5% bootstrap quantiles. Scalar metric points show bootstrap means with 2.5% to 97.5% bootstrap intervals; open diamonds indicate the original leave-one-out estimates.